# Day 4: Tokenization

## Goal

Understand how text becomes tokens, inspect token IDs, and see how prompt design affects token usage and estimated cost.

By the end of this notebook you will be able to:

- tokenize text with Python
- inspect token IDs
- compare prompts by token count
- estimate the cost impact of longer prompts

This is an important foundation for prompt design, budgeting, and performance tuning.

## Step 0: Sync the Project Once

The Day 4 dependency is already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

For course maintainers, the tokenizer package was added with:

```bash
uv add tiktoken
```

In [ ]:
import tiktoken

print("tiktoken loaded successfully.")

## Step 1: Pick an Encoding

A tokenizer converts text into token IDs.

For this notebook we use `cl100k_base`, which is a common modern encoding for LLM work.

In [ ]:
encoding = tiktoken.get_encoding("cl100k_base")
encoding

## Step 2: See Tokens and Token IDs

Token IDs are the integer representation of the text after tokenization.

In [ ]:
sample_text = "FastAPI makes it easy to build Python APIs."
token_ids = encoding.encode(sample_text)

print("Text:")
print(sample_text)
print()
print("Token IDs:")
print(token_ids)
print()
print("Token count:", len(token_ids))

In [ ]:
decoded_pieces = [encoding.decode([token_id]) for token_id in token_ids]
list(zip(token_ids, decoded_pieces))

## Step 3: Create a Helper to Compare Prompts

This helper measures:

- number of characters
- number of tokens
- token IDs

In [ ]:
def analyze_prompt(text: str) -> dict:
    token_ids = encoding.encode(text)
    return {
        "text": text,
        "characters": len(text),
        "tokens": len(token_ids),
        "token_ids": token_ids,
    }


print("Prompt analysis helper ready.")

## Step 4: Compare Short vs Medium vs Verbose Prompts

The wording changes, but the task is basically the same. This is where token efficiency becomes visible.

In [ ]:
prompts = {
    "short": "Summarize this invoice.",
    "medium": "Summarize this invoice and list the vendor name, invoice date, and total amount.",
    "verbose": "Please carefully read the invoice below, understand all relevant commercial details, and provide a concise summary that includes the vendor name, invoice date, total amount, and any notable line items if they are visible.",
}

results = {name: analyze_prompt(text) for name, text in prompts.items()}

for name, result in results.items():
    print(name.upper())
    print("Prompt:", result["text"])
    print("Characters:", result["characters"])
    print("Tokens:", result["tokens"])
    print("First 20 token IDs:", result["token_ids"][:20])
    print("-" * 80)

## Step 5: Estimate Token Usage (No API Call)

We can estimate token usage locally with `tiktoken`.
Cost depends on live pricing and a real API call, so we do not calculate cost here.


In [ ]:
REQUESTS_PER_DAY = 1000

for name, result in results.items():
    tokens_per_request = result["tokens"]
    tokens_per_day = tokens_per_request * REQUESTS_PER_DAY
    print(name.upper())
    print(f"Tokens per request: {tokens_per_request}")
    print(f"Tokens for {REQUESTS_PER_DAY} requests/day: {tokens_per_day}")
    print("-" * 80)


## Step 6: Get Cost from a Real API Call (LangChain Callback)

If you call the model, LangChain can report token usage and cost via `get_openai_callback()`.
This avoids hardcoding prices in the notebook.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

try:
    from langchain.callbacks import get_openai_callback
except Exception:
    from langchain_community.callbacks import get_openai_callback

load_dotenv(override=True)

model_name = os.getenv("OPENAI_MODEL")
if not model_name:
    raise ValueError("OPENAI_MODEL is not set in .env")

llm = ChatOpenAI(model=model_name, temperature=0)
question = "Explain tokenization in one simple sentence."
messages = [HumanMessage(content=question)]

with get_openai_callback() as cb:
    response = llm.invoke(messages)

    print(response.content)
    print(f"Prompt tokens: {cb.prompt_tokens}")
    print(f"Completion tokens: {cb.completion_tokens}")
    print(f"Total tokens: {cb.total_tokens}")
    print(f"Total cost (USD): ${cb.total_cost}")
    print(f"Cost: ${cb.total_cost:.6f} | Tokens: {cb.total_tokens}")


## Step 7: Compare Two Realistic Prompt Styles

This is closer to what happens in applications.

One version is minimal. The other version adds extra explanation and formatting instructions.

In [ ]:
prompt_a = "Extract vendor, total amount, and invoice date from this invoice."

prompt_b = "You are an expert finance assistant. Carefully read the invoice text below and extract the vendor name, total amount, and invoice date. Return the answer clearly and do not add extra explanation unless the value is missing."

analysis_a = analyze_prompt(prompt_a)
analysis_b = analyze_prompt(prompt_b)

print("Prompt A tokens:", analysis_a["tokens"])
print("Prompt B tokens:", analysis_b["tokens"])
print("Extra tokens in Prompt B:", analysis_b["tokens"] - analysis_a["tokens"])

## Day 4 Recap

You saw that:

- text is converted into token IDs
- longer prompts usually mean more tokens
- more tokens usually mean higher cost
- prompt design affects efficiency, not just model behavior

That is why tokenization matters before building larger RAG and agent systems.